# CPU notebook 5: early-signal test, prediction targets and budget controller

Runs the analyses for the early-signal test, the comparison of prediction targets and the cold-start budget
controller with the released code, three seeds where a model is trained, on the same type of CPU instance as
notebooks 3 and 4.

Run in two parts so that each stays well within a 12-hour session; set `PART` in cell 3.
* `PART = "A"`: early-signal test and controls (both traces), budget controller (three seeds), targets at default
  settings (three seeds), and the figures that need the prepared traces.
* `PART = "B"`: targets with a 10 s hold and with doubled cold-start durations (three seeds each).

**Setup.** Attach the code dataset (this repository) and the data dataset; accelerator None (CPU).

In [ ]:
# 1. Find the code and data (any depth under /kaggle/input). No pip install: the code is put on Python's path.
import glob, os, shutil, sys
print('looking for the code ...', flush=True)
hits = [p for p in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(p), 'src', 'stawarm'))]
print('found:', hits, flush=True)
assert hits, 'Code not found. Check that the stawarm-code dataset is attached (Add Input) and contains pyproject.toml and src/.'
# Use the copy of this repository (it has reproduce.py); older copies in the same dataset are ignored.
current = [p for p in hits if os.path.exists(os.path.join(os.path.dirname(p), 'reproduce.py'))]
assert current, 'The attached stawarm-code dataset holds an older version of the code; upload the current repository as a new version.'
code_dir = os.path.dirname(current[0])
WORK = '/kaggle/working/stawarm'
if not os.path.exists(WORK):
    shutil.copytree(code_dir, WORK)
os.chdir(WORK)
# The repository ships the paper's result files in results/; keep them aside for comparison so that this
# notebook writes only its own outputs to results/.
if os.path.isdir('results') and not os.path.isdir('results_paper'):
    shutil.move('results', 'results_paper')
os.makedirs('results', exist_ok=True)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')          # used by every !python command below
sys.path.insert(0, os.path.join(WORK, 'src'))
os.makedirs('data/processed', exist_ok=True)
for name in ['azure2021_events.parquet', 'huawei_r2_day30_events.parquet', 'huawei_r2_cold_start_stats.parquet']:
    src = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    print(name, '->', src, flush=True)
    assert src, f'{name} not found: check that the stawarm-data dataset is attached'
    if not os.path.exists(f'data/processed/{name}'):
        os.symlink(src[0], f'data/processed/{name}')
import stawarm
print('code from', code_dir, '| package OK:', stawarm.__file__)
print('data:', os.listdir('data/processed'), flush=True)

In [ ]:
# 2. Record the software and hardware environment of this run
!python scripts/tools/record_environment.py results/environment_05_cpu_early_signal_targets_budget.txt

In [ ]:
# 3. Tests (should end with '35 passed')
!python -m pytest -q 2>&1 | tail -2

In [ ]:
# 4. Which part to run: "A" or "B"
PART = "A"
THETAS = "0.02,0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.6,0.7,0.8,0.9"
print("part", PART)

## 5. Part A: early-signal test and controls (model-free)

In [ ]:
if PART == "A":
    !python -u scripts/experiments/run_early_signal_test.py
    !python -u scripts/experiments/run_early_signal_controls.py
    !python -u scripts/experiments/run_early_signal_controls.py --dataset huawei_day30

## 6. Part A: budget controller, three seeds

In [ ]:
if PART == "A":
    for seed in [0, 1, 2]:
        !python -u scripts/experiments/run_budget_controller.py --model nolead --lead-feature --seed {seed} --tag s{seed} 2>&1 | grep -v Warning

## 7. Part A: prediction targets at default settings, three seeds

In [ ]:
if PART == "A":
    for seed in [0, 1, 2]:
        !python -u scripts/experiments/run_policies.py --variants app_nolead,app_lead,app_union --risk-on none --lead-feature --thetas {THETAS} --seed {seed} --tag targets_s{seed} 2>&1 | grep -v Warning
    !python scripts/analysis/summarize_runs.py "results/policies_azure2021_targets_s*" --out results/summary_targets_azure.txt

## 8. Part B: prediction targets with a 10 s hold and with doubled cold-start durations, three seeds

In [ ]:
if PART == "B":
    for seed in [0, 1, 2]:
        !python -u scripts/experiments/run_policies.py --horizon 10 --variants app_nolead,app_lead,app_union --risk-on none --lead-feature --thetas {THETAS} --seed {seed} --tag targets_s{seed} 2>&1 | grep -v Warning
    for seed in [0, 1, 2]:
        !python -u scripts/experiments/run_policies.py --scale 2 --variants app_nolead,app_lead,app_union --risk-on none --lead-feature --thetas {THETAS} --seed {seed} --tag targets_s{seed} 2>&1 | grep -v Warning
    !python scripts/analysis/summarize_runs.py "results/policies_azure2021_h10_targets_s*" --out results/summary_targets_h10_azure.txt
    !python scripts/analysis/summarize_runs.py "results/policies_azure2021_x2_targets_s*" --out results/summary_targets_x2_azure.txt

## 9. Figures that need the prepared traces or the outputs above (part A)

In [ ]:
if PART == "A":
    !python scripts/analysis/make_figures.py

In [ ]:
# 10. Compare this run with the paper's result files (zero differences expected for tree-based runs)
!python scripts/analysis/compare_results.py results results_paper --out results/comparison_05_cpu_early_signal_targets_budget.txt

In [ ]:
# 11. Pack the results for download (appears under Output)
import shutil
shutil.make_archive(f'/kaggle/working/results_cpu_rerun_part{PART}', 'zip', 'results')
print('packed results -> results_cpu_rerun_part' + PART + '.zip')